# 🔐 HR PII Masking Gateway Hands-on

이번 노트북을 통해서는 Amazon Bedrock AgentCore GateWay를 활용한 세분화된 접근 제어 및 인터셉터 활용에 대한 실습을 진행하고자 합니다.

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">전체 시스템 아키텍처</h4>
<img src="generated-diagrams/01-overall-architecture-overview.png" alt="전체 시스템 아키텍처" style="max-width:100%;height:auto;" />
</div>

### 배포되는 AWS 리소스
| 리소스 | 설명 |
|--------|------|
| Bedrock Guardrails | PII 탐지 및 익명화 |
| Amazon Cognito | JWT 기반 사용자 인증 |
| DynamoDB (3개 테이블) | 권한, 직원 데이터, 휴가 기록 |
| Lambda (6개 함수) | 인터셉터 2개 + 도구 4개 |
| Bedrock AgentCore Gateway | Agent 게이트웨이 |

### 전체 요청 흐름

```
Client → Cognito JWT Auth → Bedrock AgentCore Gateway
                                ├── ① REQUEST Interceptor (권한 사전 검증)
                                │     └── DynamoDB EmployeePermissions 조회
                                ├── ② MCP Tool Lambdas (4개 HR 도구)
                                │     └── DynamoDB Employees/LeaveRecords 조회
                                └── ③ RESPONSE Interceptor (PII 마스킹 + FGAC)
                                      ├── Bedrock Guardrail (Phone, Address 마스킹)
                                      └── Code-level (추가적인 민감한 정보 제거)
```

---
## 1. Setup and Imports

프로젝트 모듈을 미리 로드합니다.

### 주요 모듈 구성
- `config/deployment_config.py`: 배포 ID, 리전, 리소스 이름 등 설정
- `src/modules/`: AWS 서비스별 설정 모듈 (Bedrock, Cognito, DynamoDB, Lambda, Gateway)
- `src/interceptors/`: REQUEST/RESPONSE 인터셉터 Lambda 코드
- `src/tools/`: HR 도구 Lambda 코드

In [ ]:
# 필요한 패키지 설치
!pip3 install -q -r requirements.txt
print('✓ 패키지 설치 완료')

In [ ]:
import sys
import os
import json
import importlib
from pathlib import Path
import boto3

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
os.chdir(NOTEBOOK_DIR)

project_root = NOTEBOOK_DIR / 'pii-masking-gateway'
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.deployment_config import *
from src.modules.bedrock_setup import setup_bedrock_guardrails
from src.modules.cognito_setup import setup_cognito_with_multiple_clients
from src.modules.dynamodb_setup import (
    setup_employee_permissions_table,
    setup_employees_table,
    setup_leave_records_table,
    load_employee_permissions,
    load_employee_data,
    load_leave_records
)
from src.modules.lambda_setup import setup_interceptor_lambda, setup_tool_lambdas, setup_request_interceptor_lambda
from src.modules.gateway_setup import setup_gateway_with_dual_interceptors, register_tools_as_targets

print("✓ Imports successful (all modules reloaded)")
print(f"✓ Project root: {project_root}")
print(f"✓ Deployment ID: {DEPLOYMENT_ID}")
print(f"✓ Region: {REGION}")

---
## 2. Initialize AWS Clients

배포에 필요한 AWS 클라이언트를 초기화합니다.

In [ ]:
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
cognito_client = boto3.client('cognito-idp', region_name=REGION)
sts_client = boto3.client('sts')

account_id = sts_client.get_caller_identity()['Account']
print(f"✓ AWS Account ID: {account_id}")
print(f"✓ Region: {REGION}")

---
## 3. Setup Bedrock Guardrails (PII 탐지 엔진)

Amazon Bedrock Guardrails를 생성하여 PII를 탐지하고 마스킹 처리합니다.

```

In [ ]:
print("Setting up Bedrock Guardrails for PII detection...")

pii_config = get_sensitive_information_policy_config()
guardrail_id, guardrail_version, guardrail_arn = setup_bedrock_guardrails(
    GUARDRAIL_NAME, pii_config, REGION
)

print(f"✓ Guardrail ID: {guardrail_id}")
print(f"✓ Guardrail Version: {guardrail_version}")
print(f"✓ Guardrail ARN: {guardrail_arn}")
print(f"✓ PII Types: {len(pii_config['piiEntitiesConfig'])}")

---
## 4. Setup Amazon Cognito (인증)

Cognito User Pool을 생성하고 2개의 앱 클라이언트를 등록합니다.

### 인증 구조
```
Cognito User Pool
  ├── Resource Server: hr-gateway (scope: tools)
  ├── App Client: hr-manager   → OAuth2 Client Credentials
  └── App Client: employee     → OAuth2 Client Credentials
```

### 설정 항목

| 항목 | 설명 |
|------|------|
| User Pool | 인증의 기반이 되는 사용자 저장소 |
| Resource Server | `hr-gateway/tools` 스코프 정의 (API 접근 권한) |
| App Client (hr-manager) | HR 관리자용 M2M 클라이언트 (`client_credentials` flow) |
| App Client (employee) | 일반 직원용 M2M 클라이언트 (`client_credentials` flow) |
| Domain | OAuth2 토큰 엔드포인트용 Cognito 도메인 |

> **인증 방식**: OAuth2 Client Credentials Flow — `client_id` + `client_secret`으로 JWT 토큰을 발급받아 Gateway에 전달합니다.

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">Cognito 인증 구조</h4>
<img src="generated-diagrams/02-cognito-auth-flow-overview.svg" alt="Cognito 인증 구조" style="max-width:100%;height:auto;" />
</div>

In [ ]:
print("Setting up Amazon Cognito with multiple clients...")

cognito_info = setup_cognito_with_multiple_clients(
    cognito_client, USER_POOL_NAME, RESOURCE_SERVER_ID, DEPLOYMENT_ID, REGION
)

print(f"✓ User Pool ID: {cognito_info['user_pool_id']}")
print(f"✓ Discovery URL: {cognito_info['discovery_url']}")
print(f"\nClients created:")
for name, info in cognito_info['clients'].items():
    print(f"  - {name}: {info['client_id']}")
    print(f"    Description: {info['description']}")

---
## 5. Setup DynamoDB Tables (데이터 저장소)

3개의 DynamoDB 테이블을 생성합니다.

### 테이블 구조

#### 1. EmployeePermissions (직원별 권한 관리)
| 필드 | 타입 | 설명 |
|------|------|------|
| `EmployeeID` (PK) | String | 직원 고유 ID (EMP-001) |
| `Role` | String | `hr-manager` 또는 `employee` |
| `AllowedTools` | List | 사용 가능한 도구 목록 |
| `Name` | String | 직원 이름 |
| `Department` | String | 부서 |
| `Username` | String | 웹 데모 로그인 ID |

이 테이블이 **도구 접근 권한의 유일한 권한 소스(Single Source of Truth)**입니다.

#### 2. Employees (직원 마스터 데이터)
| 필드 | 타입 | 설명 |
|------|------|------|
| `employee_id` (PK) | String | 직원 고유 ID |
| `name`, `department`, `position` | String | 기본 정보 |
| `email`, `phone`, `address` | String | 연락처 (PII) |
| `salary` | Number | 연봉 (민감 정보) |

#### 3. LeaveRecords (휴가 기록)
| 필드 | 타입 | 설명 |
|------|------|------|
| `employee_id` (PK) | String | 직원 ID |
| `leave_id` (SK) | String | 휴가 기록 ID |
| `leave_type`, `start_date`, `end_date` | String | 휴가 정보 |

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">DynamoDB 데이터 모델</h4>
<img src="generated-diagrams/03-dynamodb-data-model-overview.svg" alt="DynamoDB 데이터 모델" style="max-width:100%;height:auto;" />
</div>

In [ ]:
print("Setting up DynamoDB tables...")

employee_permissions_table_name = f"EmployeePermissions-{DEPLOYMENT_ID}"

setup_employee_permissions_table(employee_permissions_table_name, REGION)
setup_employees_table(REGION)
setup_leave_records_table(REGION)

print(f"✓ Employee Permissions Table: {employee_permissions_table_name}")
print(f"✓ Employees Table: Employees")
print(f"✓ Leave Records Table: LeaveRecords")

---
## 6. Load Sample Data

### 1. 직원 권한 데이터 (EmployeePermissions)

| EmployeeID | Username | Role | AllowedTools |
|------------|----------|------|--------------|
| EMP-001 | hr-manager | hr-manager | all_employees, employee_search, all_leave_records, employee_leave, x_amz_bedrock_agentcore_search (5개) |
| EMP-002 | employee2 | employee | employee_search, employee_leave, x_amz_bedrock_agentcore_search (3개) |
| EMP-003 | employee3 | employee | employee_search, employee_leave, x_amz_bedrock_agentcore_search (3개) |

### 2. 직원 데이터 (Employees)

### 3. 휴가 기록 (LeaveRecords)

In [ ]:
print("Loading sample data into DynamoDB...")

load_employee_permissions(employee_permissions_table_name, REGION)
load_employee_data(REGION)
load_leave_records(REGION)

print("✓ Sample data loaded successfully")

---
## 7. Setup Lambda Functions

총 6개의 Lambda 함수를 배포합니다.

### Lambda 함수 구성

| 함수 | 역할 |
|------|------|
| **Request Interceptor** | 도구 접근 권한 사전 검증 |
| **Response Interceptor** | PII 마스킹 + tools/list 필터링 |
| all_employees_tool | 전체 직원 목록 조회 |
| employee_search_tool | 직원 검색 (ID/이름) |
| all_leave_records_tool | 전체 휴가 기록 조회 |
| employee_leave_tool | 특정 직원 휴가 조회 |

### Lambda 배포 소스 파일

| Lambda 함수 | 소스 파일 | 배포 함수 |
|-------------|-----------|-----------|
| Request Interceptor | `src/interceptors/request_interceptor.py` | `setup_request_interceptor_lambda()` |
| Response Interceptor | `src/interceptors/combined_interceptor.py` | `setup_interceptor_lambda()` |
| Tool Lambdas (4개) | `src/tools/*.py` | `setup_tool_lambdas()` |

In [ ]:
print("Setting up Lambda functions...")

# Response Interceptor
lambda_arn = setup_interceptor_lambda(
    LAMBDA_FUNCTION_NAME, LAMBDA_ROLE_NAME,
    guardrail_id, guardrail_version,
    REGION,
    employee_permissions_table_name=employee_permissions_table_name
)
print(f"✓ Response Interceptor ARN: {lambda_arn}")

# Request Interceptor
request_interceptor_name = f"hr-request-interceptor-lambda-{DEPLOYMENT_ID}"
request_lambda_arn = setup_request_interceptor_lambda(
    lambda_function_name=request_interceptor_name,
    lambda_role_arn=f"arn:aws:iam::{account_id}:role/{LAMBDA_ROLE_NAME}",
    deployment_id=DEPLOYMENT_ID,
    region=REGION,
    employee_permissions_table_name=employee_permissions_table_name
)
print(f"✓ Request Interceptor ARN: {request_lambda_arn}")

# Tool Lambdas
deployed_tools = setup_tool_lambdas(DEPLOYMENT_ID, REGION)
print(f"✓ Deployed {len(deployed_tools)} tool Lambda functions")
for tool in deployed_tools:
    print(f"  - {tool['function_name']}")

---
## 🔍 Deep Dive: Request Interceptor (권한 사전 검증)

Request Interceptor는 Gateway의 **REQUEST** 인터셉션 포인트에서 동작하며, 도구 실행 **전에** 권한을 검증합니다.

### 처리 흐름

```
요청 수신 → JWT 파싱 → Method 확인
  ├── tools/list → 바로 ALLOW (Response Interceptor에서 필터링)
  └── tools/call → client_id로 DynamoDB Permissions 조회
                    → tool_name이 AllowedTools에 있는지 확인
                    → ALLOW 또는 403 DENY
```

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">Request Interceptor 처리 흐름</h4>
<img src="generated-diagrams/04-request-interceptor-flow-overview.svg" alt="Request Interceptor 처리 흐름" style="max-width:100%;height:auto;" />
</div>

### 핵심 동작 원리

1. **JWT 토큰 파싱**: Authorization 헤더에서 Bearer 토큰을 추출하고 payload를 디코딩하여 `client_id`, `username` 등을 확인
2. **Method 분기**: `tools/list`는 바로 ALLOW (필터링은 Response Interceptor가 담당), `tools/call`은 권한 검증 수행
3. **DynamoDB 권한 조회**: `check_tool_permission()`에서 Permissions 테이블을 쿼리하여 해당 client의 AllowedTools 목록 확인
4. **Gateway 접두사 처리**: AgentCore Gateway는 도구 이름 충돌 방지를 위해 `{target_name}__{tool_name}` 형식으로 자동 네임스페이싱합니다 ([AWS 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-tool-naming.html)). 인터셉터에서는 `___` 구분자로 split하여 실제 tool_name을 추출합니다.
5. **응답 형식**: ALLOW 시 `transformedGatewayRequest`, DENY 시 `transformedGatewayResponse` 반환

```python
# Request Interceptor 핵심 코드
# 파일 위치: pii-masking-gateway/src/interceptors/request_interceptor.py

# ── 1. 권한 조회: DynamoDB EmployeePermissions에서 AllowedTools 확인 ──
def check_employee_tool_permission(employee_id, tool_name, request_id):
    employee_permissions_table_name = os.environ.get('EMPLOYEE_PERMISSIONS_TABLE_NAME')

    # Gateway 접두사 처리: "target-name___employee_search_tool" → "employee_search_tool"
    # AgentCore Gateway는 {target_name}__{tool_name} 형식으로 자동 네임스페이싱
    actual_tool_name = tool_name
    if '___' in tool_name:
        actual_tool_name = tool_name.split('___')[-1]

    dynamodb = boto3.resource('dynamodb')
    table = dynamodb.Table(employee_permissions_table_name)
    response = table.get_item(Key={'EmployeeID': employee_id})
    item = response.get('Item')

    allowed_tools = item.get('AllowedTools', [])
    role = item.get('Role', 'unknown')

    if actual_tool_name in allowed_tools:
        return True, {'role': role, 'allowed_tools': allowed_tools}
    return False, {'role': role, 'reason': f"Tool '{actual_tool_name}' not in allowed tools"}


# ── 2. ALLOW 응답: transformedGatewayRequest → 도구 Lambda로 전달 ──
def create_allow_response(message, request_id, request_body=None):
    return {
        'interceptorOutputVersion': '1.0',
        'mcp': {
            'transformedGatewayRequest': {
                'body': request_body if request_body else {}
            }
        }
    }


# ── 3. DENY 응답: transformedGatewayResponse → 즉시 클라이언트에 반환 ──
def create_deny_response(reason, request_id, jsonrpc_id=1):
    return {
        "interceptorOutputVersion": "1.0",
        "mcp": {
            "transformedGatewayResponse": {
                "statusCode": 200,
                "body": {
                    "jsonrpc": "2.0",
                    "id": jsonrpc_id if jsonrpc_id is not None else 1,
                    "result": {
                        "isError": True,
                        "content": [
                            {
                                "type": "text",
                                "text": json.dumps({"access_denied": True, "message": reason})
                            }
                        ]
                    }
                }
            }
        }
    }
```


### Request Interceptor 응답 형식

| 응답 유형 | 반환 키 | 동작 |
|-----------|---------|------|
| **ALLOW** | `transformedGatewayRequest` | 원본 요청 body를 도구 Lambda로 전달 |
| **DENY** | `transformedGatewayResponse` | `statusCode: 200` + `isError: true`로 즉시 클라이언트에 반환 |

> **핵심 포인트**: `transformedGatewayRequest`를 반환하면 요청이 도구로 전달되고, `transformedGatewayResponse`를 반환하면 Gateway가 즉시 해당 응답을 클라이언트에 반환합니다.

### Request Interceptor Input/Output 페이로드 형식

> 📖 참고: [AWS 공식 문서 - Types of interceptors](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-interceptors-types.html)

#### Input (Gateway → Request Interceptor Lambda)

Gateway가 Request Interceptor Lambda를 호출할 때 전달하는 페이로드입니다. `gatewayRequest`에 원본 요청 정보가 포함됩니다.

```json
{
  "interceptorInputVersion": "1.0",
  "mcp": {
    "rawGatewayRequest": {
      "body": "<raw_request_body>"
    },
    "gatewayRequest": {
      "path": "/mcp",
      "httpMethod": "POST",
      "headers": {
        "Authorization": "<bearer_token>",
        "Mcp-Session-Id": "<session_id>"
      },
      "body": {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
          "name": "target___tool_name",
          "arguments": { "employee_id": "EMP-001" }
        }
      }
    }
  }
}
```

> **참고**: `headers` 필드는 인터셉터 설정에서 `passRequestHeaders`가 `true`인 경우에만 포함됩니다.

#### Output (Request Interceptor Lambda → Gateway)

Request Interceptor는 두 가지 응답 중 하나를 반환합니다:

**Case 1: ALLOW** — `transformedGatewayRequest`를 반환하면 요청이 도구 Lambda로 전달됩니다.
```json
{
  "interceptorOutputVersion": "1.0",
  "mcp": {
    "transformedGatewayRequest": {
      "body": {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": { "name": "tool_name", "arguments": {...} }
      }
    }
  }
}
```

**Case 2: DENY** — `transformedGatewayResponse`를 반환하면 Gateway가 즉시 해당 응답을 클라이언트에 반환합니다.
```json
{
  "interceptorOutputVersion": "1.0",
  "mcp": {
    "transformedGatewayResponse": {
      "statusCode": 200,
      "body": {
        "jsonrpc": "2.0",
        "id": 1,
        "result": {
          "isError": true,
          "content": [{"type": "text", "text": "{\"access_denied\": true, \"message\": \"Access denied\"}"}]
        }
      }
    }
  }
}
```

> **중요**: `transformedGatewayResponse`가 포함되어 있으면 `transformedGatewayRequest`가 함께 있더라도 Gateway는 즉시 응답을 반환합니다.

---
## 🔍 Deep Dive: Response Interceptor (PII 마스킹 + FGAC)

Response Interceptor는 Gateway의 **RESPONSE** 인터셉션 포인트에서 동작하며, 도구 실행 결과를 가공합니다.

### 두 가지 역할

| 역할 | 트리거 | 동작 |
|------|--------|------|
| **FGAC 필터링** | `tools/list` 응답 | DynamoDB에서 AllowedTools 조회 → 허용된 도구만 반환 |
| **PII 마스킹** | `tools/call` 응답 | 역할/Self-access 판별 → Bedrock Guardrails로 PII 마스킹 |

### PII 마스킹 결정 로직

```
① 역할 판별 (_user_context → DynamoDB EmployeePermissions)
  ├── _user_context.employee_id로 get_employee_permissions() 조회
  ├── 또는 X-User-Employee-Id 헤더로 조회 (MCPClient용)
  └── 둘 다 없으면 → employee로 기본 설정 (최대 마스킹)

② 역할별 분기
  ├── hr-manager → 마스킹 없음 (전체 데이터 반환)
  └── employee → Self-access 판별
        ├── employee_id 직접 매칭 → Self-access (마스킹 없음)
        ├── 이름 검색 결과가 1건이고 본인 ID 매칭 → Self-access (마스킹 없음)
        └── 타인 데이터 조회 → PII 마스킹 적용
              ├── strip_sensitive_fields(): salary 필드 제거 (코드 레벨)
              └── mask_pii_with_guardrails(): Bedrock Guardrail 적용 (Phone, Address 등)
```

### 마스킹 결과 예시

| 필드 | 원본 | 마스킹 후 |
|------|------|----------|
| Phone | +1-555-0101 | {PHONE} |
| Address | 123 Oak Street, Boston | {ADDRESS} |
| Email | employee1@company.com | employee1@company.com (유지) |
| Salary | 85000 | (필드 자체 제거) |

### 핵심 함수 요약

| 함수 | 역할 |
|------|------|
| `extract_client_id_from_jwt()` | JWT에서 client_id 추출 |
| `get_employee_permissions()` | DynamoDB EmployeePermissions에서 직원 권한 정보 조회 (Role, AllowedTools 등) |
| `filter_tools_by_permissions()` | tools/list 응답에서 허용된 도구만 필터링 |
| `strip_sensitive_fields()` | Guardrail이 감지하지 못하는 민감 필드(salary 등) 코드 레벨에서 제거 |
| `mask_pii_with_guardrails()` | Bedrock Guardrails API로 PII 마스킹 |
| `mask_tool_response()` | 역할/Self-access 판별 후 마스킹 적용 (메인 마스킹 오케스트레이터) |

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">Response Interceptor 처리 흐름</h4>
<img src="generated-diagrams/05-response-interceptor-flow-overview.svg" alt="Response Interceptor 처리 흐름" style="max-width:100%;height:auto;" />
</div>

```python
# Response Interceptor 핵심 코드
# 파일 위치: pii-masking-gateway/src/interceptors/combined_interceptor.py

# ── 1. tools/list 필터링: 허용된 도구만 반환 ──
def filter_tools_by_permissions(tools, allowed_tools):
    filtered = []
    for tool in tools:
        tool_name = tool.get('name', '')
        # Gateway 접두사 제거 후 비교
        actual_name = tool_name.split('___')[-1] if '___' in tool_name else tool_name
        if actual_name in allowed_tools:
            filtered.append(tool)
    return filtered


# ── 2. Bedrock Guardrails PII 마스킹 ──
def mask_pii_with_guardrails(text):
    response = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source='OUTPUT',
        content=[{'text': {'text': text}}]
    )
    if response.get('action') == 'GUARDRAIL_INTERVENED':
        outputs = response.get('outputs', [])
        if outputs:
            return outputs[0].get('text', text)
    return text


# ── 3. tools/call 응답 마스킹: 역할/Self-access 판별 후 PII 마스킹 ──
def mask_tool_response(response_body, request_body=None, request_headers=None):
    # ① _user_context에서 employee_id 추출 → EmployeePermissions에서 역할 조회
    params = request_body.get('params', {})
    arguments = params.get('arguments', {})
    user_context = arguments.get('_user_context', {})
    user_employee_id = user_context.get('employee_id')

    # EmployeePermissions에서 역할 조회 (fallback: _user_context의 role)
    emp_perms = get_employee_permissions(user_employee_id)
    client_role = emp_perms.get('Role') if emp_perms else user_context.get('role', '')

    # Fallback: MCPClient는 _user_context를 보내지 않으므로 X-User-Employee-Id 헤더 사용
    if not client_role and request_headers:
        header_employee_id = request_headers.get('X-User-Employee-Id')
        if header_employee_id:
            user_employee_id = header_employee_id
            emp_perms = get_employee_permissions(header_employee_id)
            client_role = emp_perms.get('Role', 'employee') if emp_perms else 'employee'

    if not client_role:
        client_role = 'employee'  # 역할 판별 불가 시 최대 마스킹

    # ② hr-manager → 마스킹 없이 반환
    if client_role == 'hr-manager':
        return response_body

    # ③ employee → Self-access 여부 확인
    is_self_access = False
    requested_employee_id = arguments.get('employee_id')

    # 방법 1: employee_id 직접 매칭
    if requested_employee_id and requested_employee_id.upper() == user_employee_id.upper():
        is_self_access = True

    # 방법 2: 이름 검색 결과가 1건이고 본인 ID와 매칭
    if not is_self_access:
        result_content = response_body.get('result', {}).get('content', [])
        if result_content:
            parsed = json.loads(result_content[0].get('text', '{}'))
            employees = parsed.get('body', {}).get('result', {}).get('employees', [])
            if len(employees) == 1 and employees[0].get('employee_id', '').upper() == user_employee_id.upper():
                is_self_access = True

    if is_self_access:
        return response_body  # 본인 데이터 → 마스킹 없음

    # ④ 타인 데이터 → salary 필드 제거 + Bedrock Guardrails PII 마스킹 적용
    masked_response = json.loads(json.dumps(response_body))
    for content_item in masked_response.get('result', {}).get('content', []):
        text_value = content_item.get('text', '')
        parsed_data = json.loads(text_value) if isinstance(text_value, str) else text_value
        data_to_process = parsed_data.get('body', {}).get('result', parsed_data)
        data_to_process = strip_sensitive_fields(data_to_process)  # salary 제거
        masked_text = mask_pii_with_guardrails(json.dumps(data_to_process))
        content_item['text'] = masked_text  # MCP 프로토콜: text 필드는 반드시 string
    return masked_response


# ── 4. 메인 핸들러: tools/list → 필터링, tools/call → PII 마스킹 ──
def lambda_handler(event, context):
    mcp_data = event.get('mcp', {})
    gateway_response = mcp_data.get('gatewayResponse', {})
    gateway_request = mcp_data.get('gatewayRequest', {})
    response_body = gateway_response.get('body', {})
    request_body = gateway_request.get('body', {})
    request_headers = gateway_request.get('headers', {})
    method = request_body.get('method', '')

    # Notification (body가 null)은 pass-through
    if response_body is None:
        return {
            'interceptorOutputVersion': '1.0',
            'mcp': {'transformedGatewayResponse': {'body': {}, 'statusCode': gateway_response.get('statusCode', 200)}}
        }

    # Error 응답 (Request Interceptor DENY 등) pass-through
    if isinstance(response_body, dict):
        if 'error' in response_body or response_body.get('result', {}).get('isError'):
            return {
                'interceptorOutputVersion': '1.0',
                'mcp': {'transformedGatewayResponse': {'headers': gateway_response.get('headers', {}), 'body': response_body, 'statusCode': gateway_response.get('statusCode', 200)}}
            }

    if method == 'tools/list':
        # _user_context 또는 X-User-Employee-Id 헤더에서 employee_id 추출
        request_params = request_body.get('params', {})
        user_context = request_params.get('_user_context', {})
        employee_id = user_context.get('employee_id') if user_context else None
        if not employee_id:
            employee_id = request_headers.get('X-User-Employee-Id')  # MCPClient fallback

        if employee_id and 'result' in response_body and 'tools' in response_body['result']:
            emp_perms = get_employee_permissions(employee_id)
            allowed_tools = emp_perms.get('AllowedTools', [])
            response_body['result']['tools'] = filter_tools_by_permissions(
                response_body['result']['tools'], allowed_tools
            )

    elif method == 'tools/call':
        # 역할/Self-access 판별 후 PII 마스킹 적용
        response_body = mask_tool_response(response_body, request_body, request_headers)

    return {
        'interceptorOutputVersion': '1.0',
        'mcp': {
            'transformedGatewayResponse': {
                'headers': gateway_response.get('headers', {}),
                'body': response_body,
                'statusCode': gateway_response.get('statusCode', 200)
            }
        }
    }
```


### Response Interceptor Input/Output 페이로드 형식

> 📖 참고: [AWS 공식 문서 - Types of interceptors](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-interceptors-types.html)

#### Input (Gateway → Response Interceptor Lambda)

Response Interceptor는 원본 요청(`gatewayRequest`)과 도구 실행 결과(`gatewayResponse`)를 모두 전달받습니다.

```json
{
  "interceptorInputVersion": "1.0",
  "mcp": {
    "rawGatewayRequest": {
      "body": "<raw_request_body>"
    },
    "gatewayRequest": {
      "path": "/mcp",
      "httpMethod": "POST",
      "headers": {
        "Authorization": "<bearer_token>",
        "Mcp-Session-Id": "<session_id>"
      },
      "body": {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": { "name": "tool_name", "arguments": {...} }
      }
    },
    "gatewayResponse": {
      "statusCode": 200,
      "headers": { "Mcp-Session-Id": "<session_id>" },
      "body": {
        "jsonrpc": "2.0",
        "id": 1,
        "result": {
          "content": [
            { "type": "text", "text": "{\"employees\": [...]}" }
          ]
        }
      }
    }
  }
}
```

> **핵심 차이점**: Request Interceptor와 달리 `gatewayResponse` 필드가 포함되어 있어 도구 실행 결과를 확인하고 가공할 수 있습니다.

#### Output (Response Interceptor Lambda → Gateway)

`transformedGatewayResponse`에 가공된 응답을 담아 반환합니다. PII 마스킹이나 도구 필터링 결과가 여기에 반영됩니다.

```json
{
  "interceptorOutputVersion": "1.0",
  "mcp": {
    "transformedGatewayResponse": {
      "statusCode": 200,
      "headers": { "Mcp-Session-Id": "<session_id>" },
      "body": {
        "jsonrpc": "2.0",
        "id": 1,
        "result": {
          "content": [
            { "type": "text", "text": "{\"employees\": [{...마스킹된 데이터...}]}" }
          ]
        }
      }
    }
  }
}
```

> **중요**: Response Interceptor도 `transformedGatewayResponse`가 포함되면 Gateway는 즉시 해당 응답을 클라이언트에 반환합니다.

### Lambda Target Input/Output 형식

> 📖 참고: [AWS 공식 문서 - Lambda function targets](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-add-target-lambda.html)

#### Tool Definition (도구 스키마)

Gateway Target으로 등록할 때 `ToolDefinition`을 제공해야 합니다. 이 스키마가 MCP 클라이언트에 노출되는 도구 정의입니다.

```json
{
  "name": "employee_search_tool",
  "description": "Search for employees by ID or name",
  "inputSchema": {
    "type": "object",
    "properties": {
      "employee_id": {
        "type": "string",
        "description": "Employee ID (e.g., EMP-001)"
      },
      "search_name": {
        "type": "string",
        "description": "Search by name (partial match)"
      }
    },
    "required": []
  }
}
```

| 필드 | 필수 | 설명 |
|------|------|------|
| `name` | ✅ | 도구 이름 |
| `description` | ✅ | 도구 설명 (MCP 클라이언트에 노출) |
| `inputSchema` | ✅ | 입력 파라미터 스키마 |
| `outputSchema` | ❌ | 출력 스키마 (선택사항) |

#### Gateway Target 등록 코드

위 Tool Definition을 사용하여 `create_gateway_target` API로 Lambda를 Gateway Target에 등록합니다. 실제 프로젝트에서 사용하는 코드입니다 (`gateway_setup.py` → `register_tools_as_targets`):

```python
for tool in deployed_tools:
    response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=f"{tool['tool_name'].replace('_', '-')}-target",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": tool["lambda_arn"],
                    "toolSchema": {"inlinePayload": [tool["tool_definition"]]}
                }
            }
        },
        credentialProviderConfigurations=[{
            "credentialProviderType": "GATEWAY_IAM_ROLE"
        }]
    )

    target_id = response['targetId']
```

`deployed_tools`는 Lamda를 구성하는 `setup_tool_lambdas()`가 반환하는 리스트로, 각 항목은 다음 구조입니다:

```python
{
    'tool_name': 'employee_search_tool',
    'function_name': 'employee-search-tool-{deployment_id}',
    'lambda_arn': 'arn:aws:lambda:...',
    'tool_definition': { ... }  
}
```

#### Lambda Event (Gateway → Tool Lambda)

Gateway가 도구 Lambda를 호출할 때 `event`와 `context` 두 개의 객체를 전달합니다.

`event`에는 `inputSchema`의 `properties`에 정의된 파라미터 값이 직접 전달됩니다.

```python
# Gateway가 전달하는 event 예시
event = {
    "employee_id": "EMP-001",
    "search_name": "Alice"
}
```

#### Lambda Context (Gateway → Tool Lambda)

`context.client_context.custom`에는 Gateway 메타데이터가 포함됩니다.

```python
# context.client_context.custom 구조
{
    "bedrockAgentCoreMessageVersion": "1.0",
    "bedrockAgentCoreAwsRequestId": "요청 ID",
    "bedrockAgentCoreMcpMessageId": "MCP 메시지 ID",
    "bedrockAgentCoreGatewayId": "Gateway ID",
    "bedrockAgentCoreTargetId": "Target ID",
    "bedrockAgentCoreToolName": "target-name___tool_name"
}
```

| 필드 | 설명 |
|------|------|
| `bedrockAgentCoreMessageVersion` | 메시지 버전 |
| `bedrockAgentCoreAwsRequestId` | AgentCore 서비스 요청 ID |
| `bedrockAgentCoreMcpMessageId` | MCP 메시지 ID |
| `bedrockAgentCoreGatewayId` | 호출된 Gateway ID |
| `bedrockAgentCoreTargetId` | 호출된 Gateway Target ID |
| `bedrockAgentCoreToolName` | `{target_name}___{tool_name}` 형식의 전체 도구 이름 |

이를 활용하면 하나의 Lambda에서 여러 도구를 처리할 수 있습니다:

```python
def lambda_handler(event, context):
    # context에서 도구 이름 추출
    original_name = context.client_context.custom['bedrockAgentCoreToolName']
    tool_name = original_name.split('___')[-1]  # target 접두사 제거
    
    if tool_name == 'employee_search_tool':
        return search_employee(event)
    elif tool_name == 'all_employees_tool':
        return list_all_employees(event)
```

> 📖 참고: [AWS 공식 문서 - Lambda function input format](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-add-target-lambda.html#gateway-building-lambda-input)

#### Lambda Response (Tool Lambda → Gateway)

Lambda 함수는 유효한 JSON 응답을 반환해야 합니다. 이 프로젝트에서는 다음 형식을 사용합니다:

```python
# 성공 응답
return {
    "statusCode": 200,
    "body": {
        "tool": "employee_search_tool",
        "result": { "employees": [...], "results_count": 1 },
        "success": True
    }
}

# 에러 응답
return {
    "statusCode": 404,
    "body": {
        "tool": "employee_search_tool",
        "error": "No employee found with ID 'EMP-999'",
        "success": False
    }
}
```

---
## 8. Setup AgentCore Gateway

AgentCore Gateway를 생성하고 이중 인터셉터를 연결합니다.

### Gateway 생성 파라미터

```python
gateway_client.create_gateway(
    name=gateway_name,
    protocolType="MCP", 
    protocolConfiguration={
        "mcp": {"supportedVersions": ["2025-06-18", "2025-03-26"], "searchType": "SEMANTIC"}
    },
    interceptorConfigurations=[
        {   
            "interceptor": {"lambda": {"arn": request_lambda_arn}},
            "interceptionPoints": ["REQUEST"],
            "inputConfiguration": {"passRequestHeaders": True}
        },
        {   
            "interceptor": {"lambda": {"arn": response_lambda_arn}},
            "interceptionPoints": ["RESPONSE"],
            "inputConfiguration": {"passRequestHeaders": True}
        }
    ],
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": allowed_client_ids # [hr-manager, employee]
        }
    },
    roleArn=gateway_role_arn
)
```

### 핵심 설정
- `passRequestHeaders: True`: 인터셉터가 JWT Authorization 헤더에 접근 가능
- `interceptionPoints`: REQUEST는 도구 실행 전, RESPONSE는 도구 실행 후
- `CUSTOM_JWT`: Cognito Discovery URL로 JWT 자동 검증

### protocolConfiguration 설명

> 📖 참고: [MCPGatewayConfiguration API](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_MCPGatewayConfiguration.html)

| 필드 | 필수 | 설명 |
|------|------|------|
| `supportedVersions` | 선택 | MCP 프로토콜 버전 명시 (현재 `2025-03-26`, `2025-06-18` 지원) |
| `instructions` | 선택 | 도구가 호출될 때 함께 전달되는 특별한 지시사항이나 컨텍스트 (최대 2048자) |
| `searchType` | 선택 | `SEMANTIC` 설정 시 자연어 기반 도구 검색 활성화 |

이 프로젝트에서는 `supportedVersions`와 `searchType`을 지정합니다:
- `searchType`: `SEMANTIC`으로 설정하여 자연어 기반 도구 검색 활성화
- `supportedVersions` : MCP 클라이언트와 Gateway 간 통신 프로토콜 버전을 명시하는 설정

In [ ]:
print("Setting up AgentCore Gateway with dual interceptors...")

gateway_info = setup_gateway_with_dual_interceptors(
    gateway_name=GATEWAY_NAME,
    response_lambda_arn=lambda_arn,
    request_lambda_arn=request_lambda_arn,
    discovery_url=cognito_info['discovery_url'],
    allowed_client_ids=list(cognito_info['client_ids'].values()),
    region=REGION,
    gateway_client=gateway_client,
    enable_semantic_search=True
)

print(f"✓ Gateway ID: {gateway_info['gateway_id']}")
print(f"✓ Gateway URL: {gateway_info['gateway_url']}")

---
## 9. Register Tools as Gateway Targets

각 도구 Lambda를 Gateway Target으로 등록합니다.

### Target 등록 구조
```python
gateway_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="employee-search-tool-target",
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": tool_lambda_arn,
                "toolSchema": {
                    "inlinePayload": [tool_definition]  # MCP 스키마
                }
            }
        }
    },
    credentialProviderConfigurations=[{
        "credentialProviderType": "GATEWAY_IAM_ROLE"
    }]
)
```

### MCP Tool Schema 예시 (employee_search_tool)
```json
{
    "name": "employee_search_tool",
    "description": "Search for specific employees by ID or name",
    "inputSchema": {
        "type": "object",
        "properties": {
            "employee_id": {"type": "string", "description": "Employee ID (e.g., EMP-001)"},
            "search_name": {"type": "string", "description": "Search by name"}
        }
    }
}
```

In [ ]:
print("Registering tools as gateway targets...")

target_ids = register_tools_as_targets(
    gateway_client, gateway_info['gateway_id'], deployed_tools
)

print(f"✓ Registered {len(target_ids)} tools")

---
## 10. Save Deployment Information

배포된 모든 리소스 정보를 `deployment_info.json`에 저장합니다. 이 파일은 웹 데모 백엔드와 cleanup 스크립트에서 사용됩니다.

In [ ]:
print("Saving deployment information...")

deployment_info = {
    'deployment_id': DEPLOYMENT_ID,
    'region': REGION,
    'gateway_id': gateway_info['gateway_id'],
    'gateway_url': gateway_info['gateway_url'],
    'lambda_function_name': LAMBDA_FUNCTION_NAME,
    'request_interceptor_name': request_interceptor_name,
    'lambda_role_name': LAMBDA_ROLE_NAME,
    'gateway_name': GATEWAY_NAME,
    'permissions_table_name': '',
    'user_info_table_name': '',
    'employee_permissions_table_name': employee_permissions_table_name,
    'user_pool_id': cognito_info['user_pool_id'],
    'discovery_url': cognito_info['discovery_url'],
    'token_url': cognito_info['token_url'],
    'clients': cognito_info['clients'],
    'client_ids': cognito_info['client_ids'],
    'guardrail_id': guardrail_id,
    'guardrail_version': guardrail_version,
    'guardrail_arn': guardrail_arn,
    'lambda_arn': lambda_arn,
    'request_lambda_arn': request_lambda_arn,
    'target_ids': target_ids,
    'deployed_tools': [t['function_name'] for t in deployed_tools],
    'tool_role_name': TOOL_ROLE_NAME,
    'employees_table_name': 'Employees',
    'leave_records_table_name': 'LeaveRecords'
}

deployment_file = project_root / 'deployment_info.json'
with open(deployment_file, 'w') as f:
    json.dump(deployment_info, f, indent=2)

print(f"✓ Deployment information saved to: {deployment_file}")

---
## 11. Deployment Summary

In [ ]:
print("="*80)
print("DEPLOYMENT COMPLETE!")
print("="*80)
print(f"\nGateway URL: {gateway_info['gateway_url']}")
print(f"Gateway ID: {gateway_info['gateway_id']}")
print(f"Guardrail ID: {guardrail_id}")
print(f"Employee Permissions Table: {employee_permissions_table_name}")

print(f"\nClient Information:")
for name, info in cognito_info['clients'].items():
    print(f"  {name}: {info['client_id']} ({info['description']})")

print(f"\nFeatures Enabled:")
print(f"  ✓ PII Masking with Bedrock Guardrails ({len(pii_config['piiEntitiesConfig'])} PII types)")
print(f"  ✓ Fine-Grained Access Control with EmployeePermissions DynamoDB")
print(f"  ✓ Per-Employee Tool Permissions (EmployeeID-based)")
print(f"  ✓ REQUEST Interceptor for pre-validation")
print(f"  ✓ RESPONSE Interceptor for PII masking")
print(f"  ✓ Dual-layer security architecture")

print(f"\nNext Steps:")
print(f"  1. Start the HR Web Demo: notebook-all/02-hr-web-demo.ipynb")
print(f"  2. Test the gateway: python scripts/test_combined.py")
print(f"  3. To clean up resources, run: python scripts/cleanup.py")

---
## 📘 가이드: 새로운 사용자 추가

새로운 사용자를 추가하려면 **4가지 작업**이 필요합니다.

### Step 1. EmployeePermissions 테이블에 권한 추가

직원의 역할과 사용 가능한 도구를 정의합니다. Request Interceptor가 이 테이블을 조회하여 도구 접근 권한을 검증합니다.

```python
emp_perm_table = dynamodb.Table(employee_permissions_table_name)

new_employee_permission = {
    'EmployeeID': 'EMP-007',           # 고유 직원 ID
    'Role': 'employee',                 # 'hr-manager' 또는 'employee'
    'AllowedTools': [                   # 사용 가능한 도구 목록
        'employee_search_tool',
        'employee_leave_tool',
        'x_amz_bedrock_agentcore_search'
    ],
    'Name': 'Employee 7',
    'Department': 'Finance',
    'Username': 'employee7'             # 웹 데모 로그인 ID
}

emp_perm_table.put_item(Item=new_employee_permission)
```

### Step 2. Employees 테이블에 직원 데이터 추가

직원의 기본 정보와 PII 데이터를 등록합니다. MCP Tool Lambda가 이 테이블에서 직원 정보를 조회합니다.

```python
emp_table = dynamodb.Table('Employees')

new_employee_data = {
    'employee_id': 'EMP-007',
    'name': 'Employee 7',
    'department': 'Finance',
    'position': 'Financial Analyst',
    'email': 'employee7@company.com',
    'phone': '+1-555-0107',
    'address': '321 Maple Lane, Arlington, MA 02474',
    'salary': 70000,
    'hire_date': '2020-11-05',
    'status': 'Active'
}

emp_table.put_item(Item=new_employee_data)
```

### Step 3. LeaveRecords 테이블에 휴가 데이터 추가

직원의 휴가 기록을 등록합니다. 이 데이터가 없으면 휴가 조회 도구에서 해당 직원의 기록이 나타나지 않습니다.

```python
leave_table = dynamodb.Table('LeaveRecords')

sample_leave_records = [
    {
        'employee_id': 'EMP-007',
        'leave_id': 'LEAVE-2024-00701',
        'leave_type': 'Annual',
        'start_date': '2024-08-05',
        'end_date': '2024-08-09',
        'days': 5,
        'status': 'Approved',
        'year': 2024,
        'reason': 'Summer vacation',
        'approved_by': 'EMP-001',
        'approved_date': '2024-07-20'
    },
    {
        'employee_id': 'EMP-007',
        'leave_id': 'LEAVE-2024-00702',
        'leave_type': 'Sick',
        'start_date': '2024-10-15',
        'end_date': '2024-10-16',
        'days': 2,
        'status': 'Approved',
        'year': 2024,
        'reason': 'Medical appointment',
        'approved_by': 'EMP-001',
        'approved_date': '2024-10-15'
    }
]

for record in sample_leave_records:
    leave_table.put_item(Item=record)
```

### Step 4. 웹 데모 백엔드 USERS_DB 업데이트

`hr-web-demo/backend/services.py`의 `USERS_DB` 딕셔너리에 로그인 정보를 추가합니다.

```python
# hr-web-demo/backend/services.py
USERS_DB = {
    # ... 기존 사용자 ...
    "employee7": {
        "password": "finance2024!",
        "role": "employee",
        "name": "Employee 7",
        "employee_id": "EMP-007",
        "department": "Finance",
        "position": "Financial Analyst"
    }
}
```

> **참고**: `USERS_DB`는 웹 데모 전용 로그인 저장소입니다. 실제 Gateway 인증은 Cognito JWT를 통해 이루어지므로, Gateway만 사용하는 경우 Step 1~3만 수행하면 됩니다.

### 자동화 스크립트

`pii-masking-gateway/scripts/add_new_user.py` 스크립트를 사용하면 Step 1~4가 모두 자동으로 처리됩니다. 스크립트 상단의 변수를 수정한 후 실행하세요.

```bash
python pii-masking-gateway/scripts/add_new_user.py
```

### 역할별 도구 권한

| 역할 | 사용 가능한 도구 |
|------|------------------|
| `hr-manager` | `all_employees_tool`, `employee_search_tool`, `all_leave_records_tool`, `employee_leave_tool`, `x_amz_bedrock_agentcore_search` |
| `employee` | `employee_search_tool`, `employee_leave_tool`, `x_amz_bedrock_agentcore_search` |

> **중요**: EmployeePermissions의 `AllowedTools`에 도구 이름을 추가하지 않으면, Request Interceptor가 해당 도구 호출을 403으로 차단합니다.

---
## 📘 가이드: 새로운 MCP Tool 추가

새로운 도구를 Gateway Target으로 추가하려면 **4단계**가 필요합니다.

### Step 1. Lambda 함수 코드 작성

`pii-masking-gateway/src/tools/` 디렉토리에 새 도구 파일을 생성합니다.

```python
# src/tools/my_new_tool.py
import json
import boto3

def lambda_handler(event, context):
    \"\"\"새 도구의 Lambda 핸들러\"\"\"
    body = event if isinstance(event, dict) else json.loads(event)
    
    # 비즈니스 로직 구현
    result = {"message": "Hello from my new tool"}
    
    return {
        "statusCode": 200,
        "body": {
            "tool": "my_new_tool",
            "result": result,
            "success": True
        }
    }

# MCP Tool Definition (Gateway 등록 시 사용)
TOOL_DEFINITION = {
    "name": "my_new_tool",
    "description": "도구 설명",
    "inputSchema": {
        "type": "object",
        "properties": {
            "param1": {"type": "string", "description": "파라미터 설명"}
        },
        "required": ["param1"]
    }
}
```

### Step 2. Lambda 배포

```python
from src.utils.aws_utils import deploy_lambda_function, grant_gateway_invoke_permission

# Lambda 배포
tool_lambda_arn = deploy_lambda_function(
    function_name=f"my-new-tool-{DEPLOYMENT_ID}",
    role_arn=tool_role_arn,  # 기존 도구 Lambda Role 재사용
    lambda_code_path="src/tools/my_new_tool.py",
    environment_vars={'TOOL_NAME': 'my_new_tool'},
    description='My new tool description',
    region=REGION
)
```

### Step 3. Gateway Target 등록

```python
tool_definition = {
    "name": "my_new_tool",
    "description": "도구 설명",
    "inputSchema": {
        "type": "object",
        "properties": {
            "param1": {"type": "string", "description": "파라미터 설명"}
        },
        "required": ["param1"]
    }
}

response = gateway_client.create_gateway_target(
    gatewayIdentifier=gateway_info['gateway_id'],
    name="my-new-tool-target",
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": tool_lambda_arn,
                "toolSchema": {"inlinePayload": [tool_definition]}
            }
        }
    },
    credentialProviderConfigurations=[{
        "credentialProviderType": "GATEWAY_IAM_ROLE"
    }]
)
print(f"Target ID: {response['targetId']}")
```

### Step 4. EmployeePermissions에 도구 권한 추가

```python
# 기존 직원의 AllowedTools에 새 도구 추가
table = dynamodb.Table(employee_permissions_table_name)

table.update_item(
    Key={'EmployeeID': 'EMP-001'},
    UpdateExpression='SET AllowedTools = list_append(AllowedTools, :new_tool)',
    ExpressionAttributeValues={':new_tool': ['my_new_tool']}
)
```

> **중요**: EmployeePermissions의 `AllowedTools`에 도구 이름을 추가하지 않으면, Request Interceptor가 해당 도구 호출을 403으로 차단합니다.

### 웹 데모 Agent에서의 자동 도구 인식

웹 데모의 AI Chat(`agent_service.py`)은 MCPClient를 통해 Gateway에서 도구를 자동으로 발견합니다.

```python
# agent_service.py의 도구 로딩 방식
with mcp_client:
    tools = mcp_client.list_tools_sync()  # Gateway에 등록된 모든 도구 자동 로딩
    agent = Agent(model=model, tools=tools)
```

따라서 Step 1~4만 완료하면 `agent_service.py` 수정 없이 새 도구가 AI Chat에서 바로 사용 가능합니다.

> **참고**: `pii-masking-gateway/scripts/add_new_tool.py` 스크립트를 사용하면 Step 1~4가 모두 자동으로 처리됩니다.

#### 예시: 도구 정의

```python
new_tool_definition = {
    "name": "my_new_tool",
    "description": "새 도구에 대한 설명을 입력하세요",
    "inputSchema": {
        "type": "object",
        "properties": {
            "param1": {
                "type": "string",
                "description": "파라미터 설명"
            }
        },
        "required": ["param1"]
    }
}
```

> 실제 배포하려면 위 가이드의 Step 1~4를 순서대로 실행하거나, `pii-masking-gateway/scripts/add_new_tool.py` 스크립트를 사용하세요.